## Task 1: Upsert 'Changed Customers' into Silver with MERGE
- **Goal:** In ONE atomic command, UPDATE existing customers and INSERT new ones.
- **Match key:** `customer_id`
- **Before MERGE:** customer 101 = "Asha Sharma", 102 = Delhi
- **Expected after MERGE:** 101 → "Asha Verma", 102 → Bengaluru, 207 & 208 added (total rows = 7)
- **Delta Lake benefit:** ACID — the merge either fully completes or fully rolls back.

In [0]:
# Read the uploaded CSVs and save them as Delta tables
seed = (
    spark.read
    .option("header" , "true")
    .option("inferSchema" , "true")
    .csv("/Volumes/cyntexa_dev/day_8/raw/Customers/")
)

(seed.write
    .mode("overwrite")
    .saveAsTable("cyntexa_dev.day_8.silver_customers")
)

updates = (
    spark.read
    .option("header" , "true")
    .option("inferSchema" , "true")
    .csv("/Volumes/cyntexa_dev/day_8/raw/Changed Customers/")
)

updates.createOrReplaceTempView("customers_updates")

In [0]:
%sql
select * from cyntexa_dev.day_8.silver_customers;

In [0]:
%sql
SELECT * FROM customers_updates;

In [0]:
%sql
MERGE INTO cyntexa_dev.day_8.silver_customers AS target 
USING customers_updates AS source 
ON target.customer_id = source.customer_id

WHEN MATCHED THEN 
UPDATE SET * 
WHEN NOT MATCHED THEN 
INSERT *;

-- Verify the upsert worked
SELECT * FROM cyntexa_dev.day_8.silver_customers;

## Task 2: Grant Access to Two Groups (Unity Catalog)
| Group | Gets | Why |
|---|---|---|
| `data_analysts` | SELECT on the **real table** | Trusted team, needs real emails for support |
| `marketing` | SELECT on **masked view** + **limited view** | Should NOT see personal contact data |

- **Masked view:** email shown as `***@email.com`
- **Limited view:** only safe columns + rows for 3 cities
- Permissions enforced at query time by Unity Catalog — no data copies needed.

In [0]:
%sql
-- MASKED VIEW: real list, but emails are hidden
CREATE OR REPLACE VIEW cyntexa_dev.day_8.v_customers_masked AS 
SELECT 
customer_id,
name,
city,
concat('***@' , split(email , '@')[1]) AS email,
phone
FROM cyntexa_dev.day_8.silver_customers;

-- LIMITED VIEW: safe columns only + row filter
CREATE OR REPLACE VIEW cyntexa_dev.day_8.v_customers_limited AS
SELECT customer_id , name , city 
FROM cyntexa_dev.day_8.silver_customers
WHERE city IN ('Mumbai', 'Delhi', 'Chennai');

In [0]:
%sql
SELECT * FROM cyntexa_dev.day_8.v_customers_masked;

In [0]:
%sql
SELECT * FROM cyntexa_dev.day_8.v_customers_limited;

# 💡 Databricks Units (DBU) - Notes

### What is a DBU?
* **DBU** stands for **Databricks Unit**.
* It is a **normalized unit of processing power** used to measure and bill your usage on the Databricks platform.
* Think of it like **electricity for cloud computing**: instead of paying for the hardware directly, you pay for the computing power you consume per second.

---

### What are you actually paying for with DBUs?
When you see a DBU charge, you are paying for the **software layer and management features** provided by Databricks, including:

1. **Processing & Engine Optimizations:** Performance enhancements like Delta Engine, photon engine execution, and high-speed query processing.
2. **Platform Management:** Automatic cluster creation, auto-scaling, security features, workspace isolation, and task scheduling.
3. **Workload Type:** Different types of jobs cost different DBU rates per hour:
   * **Automated / Jobs Compute:** Cheaper (used for scheduled, non-interactive batch jobs).
   * **All-Purpose Compute:** Higher cost (used for interactive notebook development, analysis, and debugging).
   * **SQL Warehouses:** Specialized rates for data warehousing and dashboard queries.

> ⚠️ **Important Note:** DBUs only cover the **Databricks software usage**. You pay your cloud provider (AWS, Azure, or GCP) separately for the underlying infrastructure (virtual machines, storage, and network bandwidth).

---

### Key Formulas & How It Works
* **Total DBU Cost** = `(Number of Nodes) × (DBU Rate per Machine Type) × (Hours Running)`
* **Total Databricks Bill** = `(Total DBUs Consumed) × (Price per DBU based on your subscription tier)`

---

### How to Check DBU Usage in a Notebook / Cluster
1. **Compute Tab:** Go to **Compute** → select your active cluster → check the **Metrics** or **Event Log** tab to see your cluster's DBU rating per hour.
2. **System Tables (Admin Query):** You can query system usage logs directly in SQL:
   ```sql
   SELECT 
       account_id,
       workspace_id,
       sku_name,
       usage_quantity AS dbus_consumed,
       usage_start_time
   FROM system.billing.usage
   ORDER BY usage_start_time DESC
   LIMIT 10;

# Intermediate Tasks: SCD Type 2 Dimension
---
## What is SCD Type 2?
Instead of overwriting a customer's address, we **keep the old row** (close it with an
`end_date` + `is_current = false`) and **insert a brand-new row** with the new address.
Result: we can answer "where did this customer live on any past date?"

In [0]:
# Load the existing SCD2 history into a Delta table
dim = spark.read.option("header" , True).csv("/Volumes/cyntexa_dev/day_8_intermediate/raw/Customers/")
dim = dim.withColumn("is_current" , dim["is_current"].cast("boolean")) # convert string -> boolean
dim.write.mode("overwrite").saveAsTable("cyntexa_dev.day_8_intermediate.dim_customers")

In [0]:
%sql
SELECT * FROM cyntexa_dev.day_8_intermediate.dim_customers;

In [0]:
# Load today's incoming changes
chg = spark.read.option("header" , True).csv("/Volumes/cyntexa_dev/day_8_intermediate/raw/Changed Customers/").createOrReplaceTempView("customer_changes")

In [0]:
%sql
SELECT * FROM customer_changes;

## Task 4: SCD Type 2 MERGE — Close Old Versions, Insert New Ones

**The 3 rules this MERGE must handle:**
| Incoming record | Action |
|---|---|
| Customer exists + tracked column CHANGED | Close old row (`end_date = today`, `is_current = false`) + insert new version |
| Customer exists + NO change | Do nothing |
| Brand-new customer | Insert directly as version 1 |

**Trick used:** the `USING` clause joins incoming changes to the **current** rows only,
so the MERGE only "sees" customers where something actually changed.

# SCD Type 2 MERGE Pipeline Documentation

## 1. Overview
This SQL query implements a **Slowly Changing Dimension Type 2 (SCD Type 2)** pattern using a single `MERGE INTO` statement. It maintains customer history by closing old records and inserting new versions when attributes (`address` or `city`) change, while preserving unchanged and existing historical records.

---

## 2. Query Architecture

### **Phase 1: Virtual Source Dataset Construction (`USING` clause)**
The source table `s` is created dynamically using a `UNION ALL` approach:

1. **Part 1 (Direct Records):** 
   * Fetches all incoming customer changes from `customer_changes`.
   * Assigns `merge_key = customer_id`.
2. **Part 2 (Changed Record Duplicates):** 
   * Joins incoming changes with current target records (`is_current = true`).
   * Filters for records where `address` or `city` has changed.
   * Assigns `merge_key = NULL` to force a `NOT MATCHED` condition during the MERGE phase.

---

### **Phase 2: Execution Rules**

| Scenario | Condition Matched | Action Taken |
| :--- | :--- | :--- |
| **Changed Customer (Old Record)** | `WHEN MATCHED AND (t.address <> s.address OR t.city <> s.city)` | Updates old active row: sets `end_date = current_date()` and `is_current = false`. |
| **Changed Customer (New Record)** | `WHEN NOT MATCHED` (via `merge_key = NULL`) | Inserts new version: sets `start_date = current_date()`, `end_date = NULL`, and `is_current = true`. |
| **Brand New Customer** | `WHEN NOT MATCHED` (New `customer_id`) | Inserts initial record: sets `start_date = current_date()`, `end_date = NULL`, and `is_current = true`. |
| **Unchanged Customer** | Matched, but fails `WHEN MATCHED AND (...)` filter | **No Action.** Active record remains untouched (`is_current = true`). |

---

## 3. Data Flow Example

Given the input data:
* **Customer 201:** Address changed (`45 FC Road` -> `77 Baner Road`).
* **Customer 203:** No changes (`22 Marine Drive`).
* **Customer 205:** Brand new customer.

### Input to Virtual Source (`s`):
| customer_id | name | address | city | merge_key | Origin |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **201** | Ravi Patel | 77 Baner Road | Pune | **201** | Part 1 |
| **203** | Omar Sy | 22 Marine Drive | Mumbai | **203** | Part 1 |
| **205** | Nina Kapoor | 5 Park Street | Kolkata | **205** | Part 1 |
| **201** | Ravi Patel | 77 Baner Road | Pune | **`NULL`** | Part 2 (Duplicate for Insert) |

### Final Pipeline Execution Summary:
* **Customer 201:** Row 1 expires active version (`is_current = false`). Row 4 inserts new version (`is_current = true`).
* **Customer 203:** Evaluated, fails update filter, remains active (`is_current = true`).
* **Customer 205:** Not found in target, inserted directly as active (`is_current = true`).

In [0]:
%sql
MERGE INTO cyntexa_dev.day_8_intermediate.dim_customers AS t
USING (
    -- Naye aur Unchanged records direct aayenge
    SELECT customer_id, name, address, city, customer_id AS merge_key
    FROM customer_changes
    
    UNION ALL
    
    -- Changed records ke liye NULL merge_key bhejenge taaki INSERT trigger ho sake
    SELECT s.customer_id, s.name, s.address, s.city, NULL AS merge_key
    FROM customer_changes s
    JOIN cyntexa_dev.day_8_intermediate.dim_customers t
      ON s.customer_id = t.customer_id AND t.is_current = true
    WHERE s.address <> t.address OR s.city <> t.city
) AS s
ON t.customer_id = s.merge_key AND t.is_current = true

-- Purane record ko CLOSE TABHI karo jab actual me Data Badla ho!
WHEN MATCHED AND (t.address <> s.address OR t.city <> s.city) THEN
  UPDATE SET end_date = current_date(), is_current = false
  
-- Naye record ko INSERT karo
WHEN NOT MATCHED THEN
  INSERT (customer_id, name, address, city, start_date, end_date, is_current)
  VALUES (s.customer_id, s.name, s.address, s.city, current_date(), NULL, true);

In [0]:
%sql
SELECT * FROM cyntexa_dev.day_8_intermediate.dim_customers;

## Task 5: Point-in-Time Question — "What was customer 201's address as of March 1st?"

**Logic:** a version was "alive" on a date if that date falls between its
`start_date` and `end_date`. Open versions have `end_date = NULL`, so we
substitute a far-future date (`9999-12-31`) for comparison.

**Expected answer:** `45 FC Road, Pune` — version 2 was alive from Feb 15
until Sep 9, so on March 1st it was the active one.

In [0]:
%sql
SELECT customer_id,
name,
address,
city,
start_date,
end_date
FROM cyntexa_dev.day_8_intermediate.dim_customers
WHERE customer_id = 201
AND DATE '2026-03-01' BETWEEN start_date AND COALESCE(end_date, DATE '9999-12-31');

### Task 6: DBU Cost Comparison & Recommendation for Nightly Pipeline

#### Cost Comparison

| Cluster Type | Typical DBU Rate / Hour | Main Use Case | Suitability for Nightly Job |
| :--- | :--- | :--- | :--- |
| **All-Purpose Cluster** | **High** (~$0.40 - $0.55 / DBU) | Interactive analysis, development, ad-hoc queries | ❌ **Expensive & Inefficient** (Requires manual termination or stays idle) |
| **Job Compute Cluster** | **Low** (~$0.15 - $0.20 / DBU) | Automated production workflows & scheduled pipelines | ✅ **Cost-Effective** (Spins up on schedule, runs job, terminates immediately) |

---

#### Key Differences in Simple Words

1. **DBU Rate Discount:** Job compute is **3x to 4x cheaper per DBU** compared to All-Purpose compute for the exact same underlying hardware.
2. **Lifecycle Management:** 
   * **All-Purpose:** Runs continuously unless manually stopped or timed out, incurring idle cost.
   * **Job Compute:** Created automatically when the nightly schedule triggers and terminates as soon as the pipeline finishes.

---

#### Recommendation for Cyntexa

> **Recommendation:** Cyntexa **must use Job Compute** for its scheduled nightly pipeline.

**Why?**
* **Massive Cost Savings:** Reduces DBU consumption and cloud costs by up to **70%–75%**.
* **Zero Idle Cost:** Ensures Cyntexa only pays for the exact minutes required to process the nightly data.
* **Production Best Practice:** Keeps development work isolated from production pipeline runs.

## Task 7: Governance Model for Cyntexa

### 1. Which data is sensitive (PII)?
PII = anything that can identify a real person.
| Table | Sensitive columns (PII) | Why |
|---|---|---|
| customers | email, phone, address, name | Directly identifies a person |
| orders | customer_id (links to PII) | Indirectly identifies a person |
| payments | card_number, bank details | Financial PII — most sensitive |
| products | none | No personal data |

### 2. Who gets access (Unity Catalog groups)?
| Group | Access | Rule |
|---|---|---|
| data_engineers | Full table (raw) | They build pipelines, need real data |
| data_analysts | Full customers table | Trusted team, needs real emails for support |
| marketing | Masked view only | Sees city & purchase history, emails hidden |
| support_team | Limited view only | Sees only customers in their region |
| interns/external vendors | No direct access | Must go through views with row filters |

**Principle: give the least access needed to do the job. No one gets "SELECT *" by default.**

### 3. How do we audit access after the fact?
- **Turn on audit logging** (workspace admin → audit logs) — records WHO queried WHAT and WHEN
- Query `system.access.audit` to check: did marketing ever try to read the raw table?
- **Monthly review:** run a grants report (`SHOW GRANTS` on all sensitive tables) to catch extra permissions that were never removed
- **Alert rule:** if anyone outside data_engineers queries payments table → alert to the security team

### Tradeoff I chose (and why)
Masked views add a small performance cost (tiny extra compute per query),
but keeping ONE copy of data with views on top is far safer than making
separate "safe copies" — copies always get out of sync and leak.

### Task 8 : Same As Task 4 (Already Handled The Edge Case In Task 4)

# 7. Customer Retention & Churn Report (Point-in-Time Correctness)

### Why a "Current State" Table Gives the WRONG Answer
A simple current-state table (where `is_current = true`) only contains the **latest status** of every customer as of today. If you attempt to calculate historical churn (e.g., *How many active customers did we have in January 2026?*) using only current records, your numbers will be inaccurate due to **Survivorship Bias**:

* **Past status is lost:** A customer who was active in January 2026 but churned in February will show up as `Churned` in the current-state table. Looking back, you would incorrectly count them as churned in January.
* **Under-counting historical active base:** Since churned customers are marked inactive today, querying the current state for past months under-counts your past active customer base. This causes artificially inflated churn rates for historical periods.

### The Solution: Point-in-Time (PIT) Reconstruction using SCD Type 2
By using the full SCD Type 2 history table with `start_date` and `end_date` bounds, we can **travel back in time** to determine a customer's exact status on any specific date.


# Point-in-Time Customer Retention & Churn Analysis

## 1. Executive Summary

This documentation explains how we calculate monthly **Active Customers**, **Churned Customers**, **Retention Rate**, and **Churn Rate** using an **SCD Type 2** history table (`cyntexa_dev.day_8_intermediate.dim_customers`).

Unlike a simple "current state" table, this approach looks at the data **point-in-time** (exactly as it was on the last day of each month).

---

## 2. Are We Calculating Churn Monthly?

**Yes!** We calculate churn on a **monthly snapshot basis**.

* On the last day of every month (`snapshot_date`), we take a snapshot of all active customers.
* We then check if those active customers closed or expired (`end_date`) within the **next 30 days** (i.e., during the following month).
* If a customer's record ended within those 30 days, they are counted as **Churned** for that month.

---

## 3. Mathematical Formulas

### A. Retained Customers
Retained Customers = Active Customers - Churned Customers

### B. Churn Rate (%)
Churn Rate (%) = (Churned Customers \ Active Customers) * 100

### C. Retention Rate (%)
Retention Rate (%) = (Retained Customers \ Active Customers) * 100

---

## 4. Query Architecture & Step-by-Step Breakdown

### **Step 1: `monthly_spine` (Calendar Backbone)**
* **What it does:** Generates a list of month-end dates for the entire year 2026 (`2026-01-31`, `2026-02-28`, etc.).
* **Why we need it:** It provides fixed reference dates to check customer status historically.

### **Step 2: `point_in_time_customers` (Historical Status Lookup)**
* **What it does:** Joins each month-end date with the SCD Type 2 table where:
  * `snapshot_date >= start_date` AND
  * `snapshot_date < COALESCE(end_date, '9999-12-31')`
* **Why we need it:** It finds which customers were actually active on that exact day, including their correct location at that time.

### **Step 3: `monthly_metrics` (Counting Active and Churned Users)**
* **Active Customers:** Counts all unique `customer_id`s present on the snapshot date.
* **Churned Customers:** Counts customers whose `end_date` falls between `snapshot_date` and `snapshot_date + 30 days`.

### **Step 4: Final Output (Percentage Calculation)**
* Calculates final retained count, churn percentage, and retention percentage rounded to 2 decimal places.

---



In [0]:
%sql
WITH monthly_spine AS (
  -- 1. Generate month-end snapshot dates for analysis (e.g., year 2026)
  SELECT explode(sequence(DATE '2026-01-31', DATE '2026-12-31', INTERVAL 1 MONTH)) AS snapshot_date
),

point_in_time_customers AS (
  -- 2. Join spine with SCD Type 2 table to get customer state at EXACT point-in-time
  SELECT 
    s.snapshot_date,
    c.customer_id,
    c.city,
    c.start_date,
    c.end_date,
    c.is_current
  FROM monthly_spine s
  JOIN cyntexa_dev.day_8_intermediate.dim_customers c
    ON s.snapshot_date >= c.start_date 
   AND s.snapshot_date < COALESCE(c.end_date, DATE '9999-12-31')
),

monthly_metrics AS (
  -- 3. Calculate active customers and churned customers per month
  SELECT 
    snapshot_date,
    city,
    COUNT(DISTINCT customer_id) AS active_customers,
    
    -- Churn Definition: Records whose end_date falls within 30 days after snapshot_date
    COUNT(DISTINCT CASE 
      WHEN end_date IS NOT NULL 
       AND end_date BETWEEN snapshot_date AND snapshot_date + INTERVAL 30 DAYS 
      THEN customer_id 
    END) AS churned_customers
    
  FROM point_in_time_customers
  GROUP BY snapshot_date, city
)

-- 4. Calculate final Retention and Churn Rates
SELECT 
  snapshot_date,
  city,
  active_customers,
  churned_customers,
  (active_customers - churned_customers) AS retained_customers,
  ROUND((churned_customers / active_customers) * 100, 2) AS churn_rate_pct,
  ROUND(((active_customers - churned_customers) / active_customers) * 100, 2) AS retention_rate_pct
FROM monthly_metrics
ORDER BY snapshot_date ASC, city ASC;